<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-13/AI_Modul_13.1_Praktikum_Integrasi_Model_dengan_Kamera.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 13.1: Praktikum Integrasi Model dengan Kamera Real-Time
**Tujuan Pembelajaran:**
1. Membangun kelas ingestion kamera multi-threaded untuk mengeliminasi latensi penumpukan buffer.
2. Mengintegrasikan model klasifikasi PyTorch dengan aliran video kontinu.
3. Merancang OSD (On-Screen Display) telemetri real-time dengan penghitung FPS dan label prediksi.

In [ ]:
import cv2
import threading
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

torch.manual_seed(42)
print(f"OpenCV Versi: {cv2.__version__} | PyTorch: {torch.__version__}")

### Bagian 1: Model Ringan dan Sintesis Stream Kamera Industri

In [ ]:
# Model Klasifikasi Cepat untuk Komputasi Tepi
class EdgeSortClassifier(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(32, num_classes)
        )
    def forward(self, x):
        return self.net(x)

model = EdgeSortClassifier(num_classes=3)
model.eval()
class_names = ['Sawit Mentah', 'Sawit Matang', 'Sawit Lewat Matang']
print("Model klasifikasi tepi siap diintegrasikan!")

### Bagian 2: Implementasi Multi-Threaded Camera Ingestion (Headless Safe)

In [ ]:
class SyntheticCameraThread:
    def __init__(self, fps=30, width=320, height=240):
        self.fps = fps
        self.width = width
        self.height = height
        self.frame = np.zeros((height, width, 3), dtype=np.uint8)
        self.lock = threading.Lock()
        self.running = False
        self.thread = None
        self.frame_count = 0
        
    def start(self):
        self.running = True
        self.thread = threading.Thread(target=self._produce, daemon=True)
        self.thread.start()
        return self
        
    def _produce(self):
        delay = 1.0 / self.fps
        while self.running:
            t_start = time.time()
            self.frame_count += 1
            # Buat frame dinamis (simulasi buah bergerak di konveyor)
            new_frame = np.full((self.height, self.width, 3), 40, dtype=np.uint8)
            # Gambar buah oranye bergerak melintasi frame
            cx = int((self.frame_count * 5) % self.width)
            cy = self.height // 2
            cv2.circle(new_frame, (cx, cy), 35, (10, 120, 220), -1) # Buah matang BGR
            
            with self.lock:
                self.frame = new_frame
                
            elapsed = time.time() - t_start
            if delay > elapsed:
                time.sleep(delay - elapsed)
                
    def read(self):
        with self.lock:
            return True, self.frame.copy()
            
    def stop(self):
        self.running = False
        if self.thread is not None:
            self.thread.join(timeout=1.0)

cam = SyntheticCameraThread(fps=30).start()
time.sleep(0.2) # Beri waktu producer berjalan
ret, test_frame = cam.read()
print(f"Pengambilan frame multi-threaded sukses: {ret}, Bentuk frame: {test_frame.shape}")

### Bagian 3: Siklus Inferensi Real-Time, Pengukuran FPS, dan Rendering OSD

In [ ]:
latencies = []
num_iterations = 40
last_processed_frame = None

for i in range(num_iterations):
    t0 = time.time()
    ret, frame = cam.read()
    if not ret:
        continue
        
    # Prapemrosesan Cepat
    crop_center = cv2.resize(frame, (64, 64))
    rgb_crop = cv2.cvtColor(crop_center, cv2.COLOR_BGR2RGB)
    tensor = torch.from_numpy(rgb_crop).permute(2, 0, 1).unsqueeze(0).float() / 255.0
    
    # Inferensi Model
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)[0]
        pred_idx = probs.argmax().item()
        conf = probs[pred_idx].item()
        
    # Pascapemrosesan & Rendering OSD
    label_text = f"{class_names[pred_idx]}: {conf*100:.1f}%"
    cv2.putText(frame, label_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    dt = (time.time() - t0) * 1000.0 # ms
    latencies.append(dt)
    fps_current = 1000.0 / dt if dt > 0 else 0
    cv2.putText(frame, f"FPS: {fps_current:.1f} | Latensi: {dt:.1f}ms", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)
    
    last_processed_frame = frame
    time.sleep(0.01)

cam.stop()
avg_latency = np.mean(latencies)
print(f"Evaluasi Selesai ({num_iterations} loop):")
print(f"  Rata-rata Latensi Inferensi: {avg_latency:.2f} ms")
print(f"  Throughput Efektif:         {1000.0 / avg_latency:.1f} FPS")

# Simpan tangkapan layar OSD terakhir
plt.figure(figsize=(6, 5))
plt.imshow(cv2.cvtColor(last_processed_frame, cv2.COLOR_BGR2RGB))
plt.title(f'Tangkapan Layar OSD Kamera Real-Time (Avg: {avg_latency:.1f}ms)')
plt.axis('off')
plt.tight_layout()
plt.savefig('praktikum_realtime_camera_integration_13_1.png', dpi=200)
plt.close()
print("[OK] Hasil tangkapan layar disimpan sebagai 'praktikum_realtime_camera_integration_13_1.png'")